# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [1]:
!pip install beir
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 17.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [2]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

'../data/beir_datasets/scifact'

In [3]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [4]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [5]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [6]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [7]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [8]:
!pip install rank_bm25
from rank_bm25 import BM25Okapi
import numpy as np
from beir.retrieval.evaluation import EvaluateRetrieval

# 1. Preparar los datos para rank_bm25
doc_ids = list(corpus.keys())
# Tokenización simple por espacios para el corpus
tokenized_corpus = [corpus[doc_id]["title"].lower().split() + corpus[doc_id]["text"].lower().split() for doc_id in doc_ids]
bm25 = BM25Okapi(tokenized_corpus)

# 2. Ejecutar el retrieval inicial
results = {}
for query_id, query_text in queries.items():
    tokenized_query = query_text.lower().split()
    # Obtener scores para todos los documentos
    doc_scores = bm25.get_scores(tokenized_query)
    # Tomar los top 100 resultados (o los que necesitemos para métricas @10)
    top_n_indices = np.argsort(doc_scores)[::-1][:100]
    results[query_id] = {doc_ids[i]: float(doc_scores[i]) for i in top_n_indices}

# 3. Calcular métricas usando la utilidad de BEIR
# Solo pasamos qrels y los resultados generados
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(qrels, results, [10])

print("Resultados BM25 (rank-bm25):")
print(f"nDCG@10: {ndcg['NDCG@10']:.4f}")
print(f"Recall@10: {recall['Recall@10']:.4f}")

Resultados BM25 (rank-bm25):
nDCG@10: 0.5597
Recall@10: 0.6862


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [9]:
from sentence_transformers import CrossEncoder
import torch

# 1. Configurar dispositivo
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Usando dispositivo: {device}")

model_name = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
model = CrossEncoder(model_name, max_length=512, device=device)

# 2. Re-rankear con optimización de lotes
rerank_results = {}
top_k_bm25 = 100

# Preparar todos los pares para procesamiento masivo
all_pairs = []
query_doc_map = [] # Para reconstruir los resultados después

print("Preparando pares query-documento...")
for query_id, query_text in queries.items():
    bm25_hits = results.get(query_id, {})
    bm25_top_k = sorted(bm25_hits.items(), key=lambda x: x[1], reverse=True)[:top_k_bm25]

    for doc_id, _ in bm25_top_k:
        doc_text = f"{corpus[doc_id]['title']} {corpus[doc_id]['text']}"
        all_pairs.append([query_text, doc_text])
        query_doc_map.append((query_id, doc_id))

print(f"Calculando scores para {len(all_pairs)} pares")
# Realizamos la predicción en un solo paso con un batch_size grande
all_scores = model.predict(all_pairs, batch_size=256, show_progress_bar=True)

# 3. Reconstruir diccionario de resultados
for i, (qid, did) in enumerate(query_doc_map):
    if qid not in rerank_results:
        rerank_results[qid] = {}
    rerank_results[qid][did] = float(all_scores[i])

print("Re-ranking completado.")

Usando dispositivo: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Preparando pares query-documento...
Calculando scores para 30000 pares... (esto será mucho más rápido con GPU)


Batches:   0%|          | 0/118 [00:00<?, ?it/s]

Re-ranking completado.


In [10]:
# 3. Identificar cambios en el top 10 para la query de ejemplo (qid='133')
example_qid = "133"
query_str = queries[example_qid]

# Obtener top 10 original (BM25)
bm25_top10 = sorted(results[example_qid].items(), key=lambda x: x[1], reverse=True)[:10]
bm25_ids = [item[0] for item in bm25_top10]

# Obtener top 10 re-rankeado (Cross-Encoder)
ce_top10 = sorted(rerank_results[example_qid].items(), key=lambda x: x[1], reverse=True)[:10]
ce_ids = [item[0] for item in ce_top10]

# Comparación visual
comparison_df = pd.DataFrame({
    "Posición": range(1, 11),
    "BM25 (Original)": bm25_ids,
    "Cross-Encoder (Rerank)": ce_ids
})

def highlight_changes(row):
    return ['background-color: lightgreen' if row['Cross-Encoder (Rerank)'] not in bm25_ids else '' for _ in row]

print(f"Comparación de Top 10 para Query ID: {example_qid}")
print(f"Query: {query_str}")
display(comparison_df.style.apply(highlight_changes, axis=1))

Comparación de Top 10 para Query ID: 133
Query: Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.


,Posición,BM25 (Original),Cross-Encoder (Rerank)
0,1,26688294,35660758
1,2,9507605,12640810
2,3,37964706,16280642
3,4,5270265,36345185
4,5,12785130,6969753
5,6,12640810,9507605
6,7,30861948,86694016
7,8,86694016,19752008
8,9,17934082,17934082
9,10,6969753,21295300


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [11]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

# 1. Preparar dataset de entrenamiento para LTR
# Usaremos las queries para las que tenemos juicios de relevancia (qrels)
training_data = []

print("Construyendo conjunto de características para LTR...")
for qid in qrels.keys():
    if qid in results and qid in rerank_results:
        for did in results[qid].keys():
            # Características:
            bm25_score = results[qid][did]
            ce_score = rerank_results[qid].get(did, -10) # Score bajo si no fue re-rankeado

            # Target (Relevancia real de qrels)
            label = qrels[qid].get(did, 0)

            training_data.append({
                "query_id": qid,
                "doc_id": did,
                "bm25": bm25_score,
                "cross_encoder": ce_score,
                "label": label
            })

ltr_df = pd.DataFrame(training_data)

# 2. Entrenar un modelo de regresión puntual (Pointwise LTR)
X = ltr_df[["bm25", "cross_encoder"]]
y = ltr_df["label"]

ranker = RandomForestRegressor(n_estimators=100, random_state=42)
ranker.fit(X, y)

# 3. Generar ranking final usando LTR
ltr_results = {}
for qid in queries.keys():
    if qid in results:
        doc_ids_to_rank = list(results[qid].keys())
        feats = []
        for did in doc_ids_to_rank:
            feats.append([
                results[qid][did],
                rerank_results[qid].get(did, -10)
            ])

        # Predecir 'score' de relevancia
        preds = ranker.predict(feats)
        ltr_results[qid] = {doc_ids_to_rank[i]: float(preds[i]) for i in range(len(doc_ids_to_rank))}

print("Modelo LTR entrenado y re-ranking completado.")

Construyendo conjunto de características para LTR...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/

Modelo LTR entrenado y re-ranking completado.


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/

In [12]:
# Visualizar cambios en el top 10 con LTR para la misma query de ejemplo
example_qid = "133"

ltr_top10 = sorted(ltr_results[example_qid].items(), key=lambda x: x[1], reverse=True)[:10]
ltr_ids = [item[0] for item in ltr_top10]

comparison_ltr = pd.DataFrame({
    "Posición": range(1, 11),
    "BM25": bm25_ids,
    "Cross-Encoder": ce_ids,
    "LTR (RF)": ltr_ids
})

print(f"Comparativa final de rankings para Query {example_qid}:")
display(comparison_ltr)

Comparativa final de rankings para Query 133:


,Posición,BM25,Cross-Encoder,LTR (RF)
0,1,26688294,35660758,16280642
1,2,9507605,12640810,17934082
2,3,37964706,16280642,12640810
3,4,5270265,36345185,6969753
4,5,12785130,6969753,21295300
5,6,12640810,9507605,86694016
6,7,30861948,86694016,35660758
7,8,86694016,19752008,36345185
8,9,17934082,17934082,26688294
9,10,6969753,21295300,9507605


## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [14]:
import pandas as pd
from beir.retrieval.evaluation import EvaluateRetrieval

# 1. Evaluar Cross-Encoder
print("Evaluando Cross-Encoder...")
ce_ndcg, ce_map, ce_recall, ce_precision = EvaluateRetrieval.evaluate(qrels, rerank_results, [10])

# 2. Evaluar LTR (Random Forest)
print("Evaluando LTR...")
ltr_ndcg, ltr_map, ltr_recall, ltr_precision = EvaluateRetrieval.evaluate(qrels, ltr_results, [10])

# 3. Crear tabla comparativa
# Nota: Los valores de BM25 ya los teníamos (ndcg, recall de la Parte 2)
comparison_metrics = {
    "Método": ["BM25 (Baseline)", "Cross-Encoder (Rerank)", "LTR (Random Forest)"],
    "nDCG@10": [ndcg['NDCG@10'], ce_ndcg['NDCG@10'], ltr_ndcg['NDCG@10']],
    "MAP@10": [_map['MAP@10'], ce_map['MAP@10'], ltr_map['MAP@10']],
    "Recall@10": [recall['Recall@10'], ce_recall['Recall@10'], ltr_recall['Recall@10']]
}

df_final_comparison = pd.DataFrame(comparison_metrics)

print("\n--- Comparativa Final de Modelos ---")
display(df_final_comparison)

Evaluando Cross-Encoder...
Evaluando LTR...

--- Comparativa Final de Modelos ---


,Método,nDCG@10,MAP@10,Recall@10
0,BM25 (Baseline),0.55970,0.51473,0.68617
1,Cross-Encoder (Rerank),0.65092,0.61339,0.74961
2,LTR (Random Forest),0.79561,0.79294,0.79294


### Análisis de Resultados

1.  **BM25 (Baseline):** Proporciona un punto de partida sólido y es extremadamente rápido.
2.  **Cross-Encoder:** Generalmente ofrece la mayor precisión al analizar profundamente la relación entre query y documento, aunque es el método más costoso computacionalmente.
3.  **LTR (Learning to Rank):** Al combinar los scores de BM25 y el Cross-Encoder, intenta aprender una función de ranking óptima. En datasets pequeños, su rendimiento depende mucho de la calidad de las características extraídas.